# Digital channel — parallel, fast resume

Same reviewed 29 methods and 79 conditions. Existing bank/results stay in place. This is an execution optimization, not a new experiment protocol.

**Stop the old runner before starting this notebook.** Use the same GPU type and same paths. Download the matching source ZIP too. Start with two workers; StegaStamp stays serial. No model CUDA run has been executed locally.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, zipfile

N_IMAGES = 100                  # After review: change ONLY this to 100; keep the same output.
MODELS = "all"                # All 29; or a list of exact method IDs for diagnostics.
MAX_HOURS = 24                # A session budget, NOT a guarantee of Colab uptime.
SSD_RESERVE_GIB = 60          # Leave SSD room for one isolated decoder environment.
PAIRWISE_TABLES = True
SOURCE_ARCHIVE = Path("COCO_Digital_Channel_PARALLEL_FAST_RESUME_source.zip")
DECODE_WORKERS = 4
ATTACK_WORKERS = 4
FULL_AUDIT = False

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if os.name != "posix":
    raise RuntimeError("Full 29-method execution requires Linux/NVIDIA: use Colab or a GPU server.")
if IN_COLAB:
    drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive") if IN_COLAB else Path("/workspace/data")
BANK_ROOT = BASE / "watermarking_production_bank_v1"       # READ ONLY: completed embedding bank.
OUTPUT_ROOT = BASE / "watermarking_digital_channel_v1"    # Durable measurements and reports.
SCRATCH_ROOT = Path("/content/wmbench_digital_scratch") if IN_COLAB else Path("/workspace/wmbench_digital_scratch")
SETTINGS = dict(n_images=N_IMAGES, models=MODELS, max_hours=MAX_HOURS, device="cuda",
    bank=str(BANK_ROOT), output=str(OUTPUT_ROOT), scratch=str(SCRATCH_ROOT),
    ssd_reserve_gib=SSD_RESERVE_GIB, pairwise=PAIRWISE_TABLES, download_author_assets=True,
    manual_assets_root=str(BANK_ROOT / "_manual_assets"))
print("Bank (read-only):", BANK_ROOT)
print("Results:", OUTPUT_ROOT)
print("Temporary fallback:", OUTPUT_ROOT / "_temporary")
print("Per method: save 78 attacked PNGs per available carrier in ZIPs → decode → commit → remove ZIPs.")

SETTINGS.update(decode_workers=DECODE_WORKERS, attack_workers=ATTACK_WORKERS, full_audit=FULL_AUDIT)


Mounted at /content/drive
Bank (read-only): /content/drive/MyDrive/watermarking_production_bank_v1
Results: /content/drive/MyDrive/watermarking_digital_channel_v1
Temporary fallback: /content/drive/MyDrive/watermarking_digital_channel_v1/_temporary
Per method: save 78 attacked PNGs per available carrier in ZIPs → decode → commit → remove ZIPs.


## Install

One small source ZIP, uploaded once and saved on Drive. No Base64. Frozen model environments remain isolated.


In [ ]:
# Upload the matching small source ZIP once; a copy is kept on Drive.
EXPECTED_SHA = "3115ddb4920c64f9e30c5e4ace91ffb521b5f449173bcbaeb863ec442b3fad94"
SAVED_ZIP = OUTPUT_ROOT / "_release" / SOURCE_ARCHIVE.name
if not SOURCE_ARCHIVE.is_file() and SAVED_ZIP.is_file():
    SOURCE_ARCHIVE = SAVED_ZIP
if not SOURCE_ARCHIVE.is_file():
    if not IN_COLAB:
        raise FileNotFoundError("Place the matching source ZIP beside this notebook or set SOURCE_ARCHIVE.")
    from google.colab import files
    uploaded = files.upload()
    matches = [Path(name) for name in uploaded if hashlib.sha256(uploaded[name]).hexdigest() == EXPECTED_SHA]
    if len(matches) != 1:
        raise ValueError("Upload exactly the source ZIP supplied with this notebook.")
    SOURCE_ARCHIVE = matches[0]
data = SOURCE_ARCHIVE.read_bytes()
if hashlib.sha256(data).hexdigest() != EXPECTED_SHA:
    raise ValueError("Source ZIP checksum mismatch. Do not use a different notebook's ZIP.")
SAVED_ZIP.parent.mkdir(parents=True, exist_ok=True)
if not SAVED_ZIP.exists():
    SAVED_ZIP.write_bytes(data)
if hashlib.sha256(SAVED_ZIP.read_bytes()).hexdigest() != EXPECTED_SHA:
    raise ValueError("Saved release readback failed")
PROJECT = SCRATCH_ROOT / "performance_release_v1"
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
    for member in archive.infolist():
        target = (PROJECT / member.filename).resolve()
        if not target.is_relative_to(PROJECT.resolve()) or member.is_dir():
            raise ValueError("Unsafe or unexpected source ZIP entry")
        content = archive.read(member)
        if target.exists() and target.read_bytes() != content:
            raise ValueError(f"Source changed: {target}. Do not overwrite it; keep bank/output paths and inspect the source conflict.")
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.8.15"], check=True)
import uv
UV = str(uv.find_uv_bin())
controller_dir = SCRATCH_ROOT / "controller"
CONTROLLER = controller_dir / "bin/python"
if not CONTROLLER.exists():
    subprocess.run([UV, "venv", "--python", "3.10.12", str(controller_dir)], check=True)
# This is the CPU attack/report + quality-metric environment, NOT a model environment.
subprocess.run([UV, "pip", "install", "--python", str(CONTROLLER),
    "numpy==1.26.4", "Pillow==10.4.0", "scipy==1.11.4", "scikit-image==0.22.0",
    "psutil==6.1.1", "gdown==5.2.0", "torch==2.5.1", "torchvision==0.20.1", "lpips==0.1.4"], check=True)
ENV = os.environ.copy()
ENV.update(PYTHONPATH=os.pathsep.join([str(PROJECT / "src"), str(PROJECT / "production/src")]),
    PYTHONUNBUFFERED="1", OMP_NUM_THREADS="4", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="4",
    PATH=str(Path(UV).parent)+os.pathsep+ENV.get("PATH", ""),
    UV_PYTHON_INSTALL_DIR=str(SCRATCH_ROOT / "python"))
CONFIG = SCRATCH_ROOT / "digital-settings.json"
CONFIG.write_text(json.dumps(SETTINGS, indent=2), encoding="utf-8")
print("Free SSD: %.1f GiB" % (shutil.disk_usage(SCRATCH_ROOT).free / 1024**3))
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)


Free SSD: 188.0 GiB


CompletedProcess(args=['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], returncode=0)

In [ ]:
# SepMark recovery preflight: read-only, three known damaged files.
from pathlib import Path
import json
sep_meta = BANK_ROOT / "sepmark/_meta"
sep_names = ["000000383838_m01", "000000419882_m04", "000000419882_m05"]
for stem in sep_names:
    row = json.loads((sep_meta / "records" / f"{stem}_native.json").read_text())
    print(stem, json.dumps({k: row[k] for k in ("signature", "output", "source_crop", "decoder_metadata", "artifacts", "execution")}, indent=2))
print("PREPARED", (sep_meta / "prepared.json").read_text())
print("JOBS", [p.name for p in (sep_meta / "jobs").glob("*.json")])
print("SIZES", [(s, (BANK_ROOT / "sepmark/native" / f"{s}.png").stat().st_size) for s in sep_names])


000000383838_m01 {
  "signature": "98155e7aa535be2ed45c4895cfbee87b1d4b0fd9d84bbc775d090334efc955f5",
  "output": "sepmark/native/000000383838_m01.png",
  "source_crop": "_inputs/crops/256/000000383838.png",
  "decoder_metadata": "sepmark/_meta/attempts/000000383838_m01_1790901020520884708/decoder.json",
  "artifacts": [
    {
      "path": "_inputs/crops/256/000000383838.png",
      "sha256": "37dfe5aee154ea94b7bc23d6cf38b6d3436e5f39a08a8b16dda2c8d61f307b35"
    },
    {
      "path": "sepmark/native/000000383838_m01.png",
      "sha256": "a44eb26509302480b12669fa55f202a952cb5bbf839e2d1df82685c64b75f452"
    },
    {
      "path": "sepmark/_meta/attempts/000000383838_m01_1790901020520884708/decoder.json",
      "sha256": "e9358aba094513348342c13345e783b9a0f71fc6bc7aace699385b3ccd910bb5"
    },
    {
      "path": "sepmark/_meta/attempts/000000383838_m01_1790901020520884708/metadata.json",
      "sha256": "ba5f44f219d6cbaa447ce3a0f3bb45dd3d34d3a1ea84a42f4741892dd8ca1087"
    }
  ],
  "

In [ ]:
# One-time SepMark environment recovery completed; do not reinstall during Run all.
# Provenance and scripts: MyDrive/watermarking_sepmark_recovery_20261016
print("SepMark recovery setup already completed; skipped.")


SepMark recovery setup already completed; skipped.


In [ ]:
# Candidate verification completed; all three original PNG hashes matched.
# Audit: MyDrive/watermarking_sepmark_recovery_20261016/candidates_v1/candidate_report.json
print("SepMark candidate verification already completed; skipped.")


SepMark candidate verification already completed; skipped.


In [ ]:
# Recovery completed: the three original SepMark PNGs are restored.
# Original SHA256 and clean decoded bits matched for all 3/3 cases.
# Full audit: MyDrive/watermarking_sepmark_recovery_20261016
print("SepMark recovery completed; no further repair is needed.")


SepMark recovery completed; no further repair is needed.


In [ ]:
# Only RoSteALS/SepWater need these author-import system libraries.
# Restore the setup used by RoSteALS_AUTHOR_FIX and RoSteALS_SepWater_RECOVERY.
# No model source, checkpoint, Python lock, payload or matching rule is changed.
import ctypes
import ctypes.util

def ensure_author_system_dependencies(models, in_colab):
    affected = {"rosteals", "sepwater"}
    selected = affected if models == "all" else affected.intersection(models)
    if not selected:
        return {}

    requirements = {
        "libmagickwand-dev": (
            ("MagickWand-6.Q16", "MagickWand-6.Q16HDRI", "MagickWand-7.Q16HDRI",
             "MagickWand-7.Q16", "MagickWand"), "MagickWandGenesis"),
        "libgl1": (("GL",), None),
    }

    def inspect_libraries():
        found = {}
        for package, (names, symbol) in requirements.items():
            for name in names:
                path = ctypes.util.find_library(name)
                if not path:
                    continue
                try:
                    library = ctypes.CDLL(path)
                    if symbol:
                        getattr(library, symbol)
                except (OSError, AttributeError):
                    continue
                found[package] = path
                break
        return found

    found = inspect_libraries()
    missing = [package for package in requirements if package not in found]
    if missing:
        install = ["apt-get", "-qq", "install", "-y", "--no-install-recommends", *missing]
        if not in_colab or not shutil.which("apt-get") or os.geteuid() != 0:
            raise RuntimeError(
                "RoSteALS/SepWater system libraries are missing. On Debian/Ubuntu run: "
                "sudo apt-get update && sudo " + " ".join(install)
                + ". Then rerun this setup cell; keep the existing output directory.")
        print("Installing missing system libraries for " + ", ".join(sorted(selected))
              + ": " + ", ".join(missing), flush=True)
        subprocess.run(["apt-get", "-qq", "update"], check=True, timeout=600)
        subprocess.run(install, check=True, timeout=600)
        found = inspect_libraries()
        missing = [package for package in requirements if package not in found]
        if missing:
            raise RuntimeError("System-library preflight failed after installation: "
                               + ", ".join(missing) + ". No model has been started.")
    print("RoSteALS/SepWater system-library preflight passed:", found, flush=True)
    return found

AUTHOR_SYSTEM_LIBRARIES = ensure_author_system_dependencies(MODELS, IN_COLAB)


Installing missing system libraries for rosteals, sepwater: libmagickwand-dev
RoSteALS/SepWater system-library preflight passed: {'libmagickwand-dev': 'libMagickWand-6.Q16.so.7', 'libgl1': 'libGL.so.1'}


## Missing psutil: telemetry-only repair

This fixes the InvisMark startup error in the parallel worker's resource probe.
If a decoder does not have psutil, the existing controller Python measures that
worker's RAM. Model environments, weights, attacks, scoring, parallel worker
limits and resume checks are unchanged. Telemetry failure still stops the run.

Use the same source ZIP and bank/results folders. No new ZIP is needed if the
original archive is already saved on Drive. Run all cells in the same GPU type.
The repair uses a separate source directory and preserves the old release.
Completed results are reused. Stop any older active runner first.


In [ ]:
# Report I/O repair v4: scientific code and previous measurements are unchanged.
import hashlib, zipfile
recovery_zip = OUTPUT_ROOT / "_release/COCO_Digital_Channel_REPORT_IO_FIX_source.zip"
assert hashlib.sha256(recovery_zip.read_bytes()).hexdigest() == "2198afa282de2c4c352700c44593d245adfa4e4e2d79194e3f0c6c205505a7f5"
PROJECT = SCRATCH_ROOT / "performance_release_report_io_v4"
with zipfile.ZipFile(recovery_zip) as archive:
    members = {n: archive.read(n) for n in archive.namelist()}
for name, data in members.items():
    target = (PROJECT / name).resolve()
    if not target.is_relative_to(PROJECT.resolve()):
        raise ValueError("Unsafe archive entry")
    if target.exists() and target.read_bytes() != data:
        raise ValueError(f"Source conflict: {target}")
for name, data in members.items():
    target = PROJECT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        target.write_bytes(data)
ENV["PYTHONPATH"] = os.pathsep.join([str(PROJECT / "src"), str(PROJECT / "production/src")])
recovery_root = BANK_ROOT.parent / "watermarking_stegastamp_recovery_20261016"
assert (recovery_root / "approved_replacement/replacement_receipt.json").is_file()
for cmd in [
    [str(CONTROLLER), "-u", "-m", "wmbench_fast.runner", "validate", "--settings", str(CONFIG)],
    [str(CONTROLLER), "-u", "-m", "stegastamp_loss_reencode", "run", "--job", str(recovery_root / "1792169597490590440_job.json"), "--work", str(recovery_root / "approved_replacement"), "--results", str(OUTPUT_ROOT)]
]:
    check = subprocess.run(cmd, env=ENV, text=True, capture_output=True)
    print(check.stdout, check.stderr, flush=True)
    check.check_returncode()
print("Report I/O v4 installed: SSD staging, verified Drive copies, previous reports archived. Ready to resume.", flush=True)


Frozen source valid: 29 methods, 100 images; no bank PNG scan
 
Approved replacement already committed; no re-embedding
 
Report I/O v4 installed: SSD staging, verified Drive copies, previous reports archived. Ready to resume.


In [ ]:
checked = subprocess.run([str(CONTROLLER), "-m", "wmbench_fast.runner", "validate", "--settings", str(CONFIG)], env=ENV, text=True, capture_output=True)
print(checked.stdout, checked.stderr)
checked.check_returncode()

Frozen source valid: 29 methods, 100 images; no bank PNG scan
 


## Start / resume

First use indexes legacy records once, with visible progress. Later resumes reuse checksummed catalogs and size/mtime inventories. Do not edit the bank or result files externally in fast mode. `FULL_AUDIT=True` rehashes the underlying evidence and is intentionally slow. Pending inputs and attack ZIPs remain checksum-checked.

The controller is detached: a browser disconnect or stopping the log-view cell does not intentionally kill it. Colab VM termination still stops work; rerun with the same paths after reconnect. This cell returns immediately while computation continues.


In [ ]:
# Stop the OLD notebook runner first. Repeating this cell will not start a duplicate fast runner.
subprocess.run([str(CONTROLLER), "-m", "wmbench_fast.launch", "start", "--settings", str(CONFIG)],
               env=ENV, check=True)


CompletedProcess(args=['/content/wmbench_digital_scratch/controller/bin/python', '-m', 'wmbench_fast.launch', 'start', '--settings', '/content/wmbench_digital_scratch/digital-settings.json'], returncode=0)

## Progress

Refresh this cell manually. Per-worker telemetry is under each method's `logs/`. Reports are generated after a method finishes. Historical canary tables are not completion evidence for 100 images.


In [ ]:
# Read-only current session status. This cell does not start or stop computation.
launch = json.loads((OUTPUT_ROOT / "_reports/fast_launcher.json").read_text())
probe = "from pathlib import Path; from wmbench.io import read_json; from wmbench_fast.launch import alive; import sys; print(alive(read_json(Path(sys.argv[1]))))"
alive_check = subprocess.run([str(CONTROLLER), "-c", probe, str(OUTPUT_ROOT / "_reports/fast_launcher.json")], env=ENV, text=True, capture_output=True, check=True)
print("CONTROLLER RUNNING:", alive_check.stdout.strip(), "PID:", launch["pid"])
receipt = BANK_ROOT / "stegastamp/_meta/recoveries/stegastamp-lost-four-user-approved-20261016.json"
repaired = json.loads(receipt.read_text())
print("STEGASTAMP REPLACEMENT:", repaired["state"], "pairs:", len(repaired["targets"]), "PNG:", sum(n.endswith(".png") for n in repaired["new_files"]))
print("Old evidence archived:", repaired["previous_bank"])
log = Path(launch["log"])
with log.open("rb") as handle:
    handle.seek(max(0, log.stat().st_size - 6500))
    print(handle.read().decode("utf-8", errors="replace"))
exit_file = Path(launch["config"]).with_name(Path(launch["config"]).stem + "_exit.json")
print("Controller exit:", exit_file.read_text() if exit_file.exists() else "not yet recorded")

CONTROLLER RUNNING: True PID: 7784
STEGASTAMP REPLACEMENT: committed pairs: 4 PNG: 16
Old evidence archived: /content/drive/MyDrive/watermarking_stegastamp_recovery_20261016/approved_replacement/previous_bank
METHOD 1/29: trustmark_q
SKIP COMPLETE trustmark_q: compact receipt + unchanged record/table fingerprints; 13.0s; no catalogue rewrite or GPU setup

Controller exit: not yet recorded


## Optional stop

Completed checkpoints remain on Drive; an interrupted case resumes from its last checkpoint. The embedding bank is read-only.


In [ ]:
# OPTIONAL. Set True only when you explicitly want to stop the background calculation.
STOP_BACKGROUND = False
if STOP_BACKGROUND:
    subprocess.run([str(CONTROLLER), "-m", "wmbench_fast.launch", "stop", "--settings", str(CONFIG)],
                   env=ENV, check=True)


In [ ]:
# Ждём уже запущенный расчёт. Ничего не перезапускаем и не удаляем.
import subprocess

monitor = r"""
import sys
import time
from pathlib import Path
from wmbench.io import read_json
from wmbench_fast.launch import alive

record = read_json(Path(sys.argv[1]) / "_reports/fast_launcher.json")
config = Path(record["config"])
exit_file = config.with_name(config.stem + "_exit.json")

if not alive(record):
    raise RuntimeError("Контроллер уже остановлен. Нужно проверить его лог.")

print("Контроллер запущен. Ожидаю завершения расчёта.", flush=True)
while alive(record):
    time.sleep(30)

if exit_file.exists():
    print("Контроллер завершился:", read_json(exit_file), flush=True)
else:
    raise RuntimeError("Процесс остановился без итогового статуса. Проверьте лог.")
"""

subprocess.run(
    [str(CONTROLLER), "-u", "-c", monitor, str(OUTPUT_ROOT)],
    env=ENV,
    check=True,
)

## Tuning and integrity

Default: two CPU package workers and up to two GPU decoders for one method. Decoder admission waits for warm-up and memory headroom. Missing GPU telemetry means sequential admission. On A100 80 GB, test `DECODE_WORKERS=4` after reviewing two-worker throughput. Memory headroom is a heuristic, not an OOM guarantee. Do not compare concurrent decode latency with old isolated latency.

The serial reference and additional workers must agree on four fixed probes; the original per-case Clean gate remains. No valid poor result is retried. Package deletion still requires original raw-result verification. Completed methods get fast receipts so their reports and environments are not rebuilt on every resume. See `PERFORMANCE.md` inside the source ZIP for the full policy and limitations.
